In [1]:
import pandas as pd
import numpy as np

CHEMIN = "../data/raw/"

train = pd.read_csv(CHEMIN + "telcopeople_train.csv")
test = pd.read_csv(CHEMIN + "telcopeople_test.csv")

print("Train :", train.shape)
print("Test :", test.shape)

Train : (5444, 42)
Test : (1507, 41)


In [11]:
def preparer_donnees(df):
    df = df.copy()

    # Nettoyage des catégories
    df["ville_affectation"] = df["ville_affectation"].str.strip().str.title()

    df["departement"] = (
        df["departement"]
        .str.strip()
        .str.upper()
        .str.replace("DEPT", "DEP", regex=False)
    )

    df["contrat"] = (
        df["contrat"]
        .str.strip()
        .str.upper()
        .str.replace(".", "", regex=False)
    )

    df["niveau_etude"] = (
        df["niveau_etude"]
        .str.strip()
        .str.upper()
        .str.replace(" ", "", regex=False)
        .str.replace("BAC2", "BAC+2", regex=False)
    )

    # Conversion des dates
    colonnes_dates = [
        "date_reference",
        "date_naissance",
        "date_embauche",
        "date_derniere_promotion",
        "date_derniere_absence"
    ]

    for colonne in colonnes_dates:
        df[colonne] = pd.to_datetime(
            df[colonne],
            format="mixed",
            dayfirst=True,
            errors="coerce"
        )

    # Conversion du salaire
    df["salaire_mensuel"] = pd.to_numeric(
        df["salaire_mensuel"]
        .astype(str)
        .str.replace("FCFA", "", regex=False)
        .str.replace(" ", "", regex=False),
        errors="coerce"
    )
        # Âge et ancienneté
    df["age"] = (
        df["date_reference"] - df["date_naissance"]
    ).dt.days / 365.25

    df["anciennete"] = (
        df["date_reference"] - df["date_embauche"]
    ).dt.days / 365.25

    # Promotion
    df["annees_depuis_promotion"] = (
        df["date_reference"] - df["date_derniere_promotion"]
    ).dt.days / 365.25

    df["jamais_promu"] = df["date_derniere_promotion"].isna().astype(int)

    # Salaire
    df["ratio_salaire_reference"] = (
        df["salaire_mensuel"] / df["reference_salaire_grade"]
    )

    df["evolution_salaire"] = (
        df["salaire_mensuel"] - df["salaire_12m_avant"]
    )

    # Engagement et performance
    df["evolution_engagement"] = (
        df["engagement_actuel"] - df["engagement_precedent"]
    )

    df["evolution_performance"] = (
        df["performance_actuelle"] - df["performance_precedente"]
    )

    # Absentéisme sur 6 mois
    df["jours_abs_6m"] = df[
        [f"jours_abs_m{i}" for i in range(1, 7)]
    ].sum(axis=1)

    df["abs_non_just_6m"] = df[
        [f"abs_non_just_m{i}" for i in range(1, 7)]
    ].sum(axis=1)

    df["nb_abs_6m"] = df[
        [f"nb_abs_m{i}" for i in range(1, 7)]
    ].sum(axis=1)
     # Indicateurs d'anomalies
    df["anciennete_invalide"] = (df["anciennete"] < 0).astype(int)

    df["salaire_anormal"] = (
    (df["salaire_mensuel"] <= 0) |
    (df["ratio_salaire_reference"] < 0.5) |
    (df["ratio_salaire_reference"] > 2)
).astype(int)
    colonnes_categorielles = df.select_dtypes(include="object").columns

    for colonne in colonnes_categorielles:
        df[colonne] = df[colonne].fillna("INCONNU")
    

    return df

In [12]:
train_prepare = preparer_donnees(train)
test_prepare = preparer_donnees(test)

print(
    "Valeurs manquantes catégorielles :",
    train_prepare.select_dtypes(include="object").isna().sum().sum()
)

Valeurs manquantes catégorielles : 0


C:\Users\DELL\AppData\Local\Temp\ipykernel_2440\2530121279.py:108: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_categorielles = df.select_dtypes(include="object").columns
C:\Users\DELL\AppData\Local\Temp\ipykernel_2440\2530121279.py:108: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/

In [9]:
train_prepare = preparer_donnees(train)
test_prepare = preparer_donnees(test)

print("Train :", train_prepare.shape)
print("Test :", test_prepare.shape)

print("\nAnomalies d'ancienneté :", train_prepare["anciennete_invalide"].sum())
print("Anomalies salariales :", train_prepare["salaire_anormal"].sum())

Train : (5444, 55)
Test : (1507, 54)

Anomalies d'ancienneté : 11
Anomalies salariales : 25


In [5]:
train_prepare = preparer_donnees(train)
test_prepare = preparer_donnees(test)

print("Train préparé :", train_prepare.shape)
print("Test préparé :", test_prepare.shape)

Train préparé : (5444, 53)
Test préparé : (1507, 52)


In [6]:
nouvelles_features = [
    "age",
    "anciennete",
    "annees_depuis_promotion",
    "jamais_promu",
    "ratio_salaire_reference",
    "evolution_salaire",
    "evolution_engagement",
    "evolution_performance",
    "jours_abs_6m",
    "abs_non_just_6m",
    "nb_abs_6m"
]

train_prepare[nouvelles_features].describe().T

,count,mean,std,min,25%,50%,75%,max
age,5444.0,36.261428,5.665205,2.053388e+01,32.369610,36.134155,40.052019,5.470773e+01
anciennete,5444.0,2.402328,1.737893,-4.490075e-01,0.985626,1.982204,3.416838,8.416153e+00
annees_depuis_promotion,950.0,1.245440,0.885854,0.000000e+00,0.583162,0.999316,1.978782,4.082136e+00
jamais_promu,5444.0,0.825496,0.379577,0.000000e+00,1.000000,1.000000,1.000000,1.000000e+00
ratio_salaire_reference,5378.0,1.017057,0.482391,0.000000e+00,0.914672,0.996377,1.076733,1.143025e+01
evolution_salaire,3983.0,54511.172483,430482.165531,-1.575000e+06,15000.000000,23000.000000,41000.000000,1.754200e+07
evolution_engagement,3571.0,-0.002378,0.374591,-1.270000e+00,-0.254845,-0.006435,0.252469,1.446501e+00
evolution_performance,3836.0,-0.004953,0.262533,-3.080000e+00,-0.150000,-0.010000,0.140000,3.600000e+00
jours_abs_6m,5444.0,1.612785,3.143741,0.000000e+00,0.000000,0.000000,2.000000,4.800000e+01
abs_non_just_6m,5444.0,0.165871,0.768336,0.000000e+00,0.000000,0.000000,0.000000,1.700000e+01


In [10]:
manquants = pd.DataFrame({
    "nombre": train_prepare.isna().sum(),
    "pourcentage": (train_prepare.isna().mean() * 100).round(2)
})

manquants = manquants[manquants["nombre"] > 0]
manquants.sort_values("pourcentage", ascending=False)

,nombre,pourcentage
date_derniere_promotion,4494,82.55
annees_depuis_promotion,4494,82.55
date_derniere_absence,3059,56.19
evolution_engagement,1873,34.40
engagement_precedent,1797,33.01
evolution_performance,1608,29.54
performance_precedente,1608,29.54
evolution_salaire,1461,26.84
salaire_12m_avant,1460,26.82
performance_actuelle,706,12.97


In [13]:
# Séparation cible et identifiants
X = train_prepare.drop(columns=["target", "id"])
y = train_prepare["target"]

X_test = test_prepare.drop(columns=["id"])

# Suppression des dates brutes
colonnes_dates = [
    "date_reference",
    "date_naissance",
    "date_embauche",
    "date_derniere_promotion",
    "date_derniere_absence"
]

X = X.drop(columns=colonnes_dates)
X_test = X_test.drop(columns=colonnes_dates)

# Vérification
print("X :", X.shape)
print("y :", y.shape)
print("X_test :", X_test.shape)
print("Colonnes identiques :", list(X.columns) == list(X_test.columns))

X : (5444, 48)
y : (5444,)
X_test : (1507, 48)
Colonnes identiques : True


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from catboost import CatBoostClassifier

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

colonnes_categorielles = X.select_dtypes(include="object").columns.tolist()

modele = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

modele.fit(
    X_train,
    y_train,
    cat_features=colonnes_categorielles,
    eval_set=(X_validation, y_validation),
    early_stopping_rounds=100
)

probabilites = modele.predict_proba(X_validation)[:, 1]

# Recherche du meilleur seuil pour le F1
meilleur_seuil = 0
meilleur_f1 = 0

for seuil in np.arange(0.10, 0.91, 0.01):
    predictions = (probabilites >= seuil).astype(int)
    score = f1_score(y_validation, predictions)

    if score > meilleur_f1:
        meilleur_f1 = score
        meilleur_seuil = seuil

print("Meilleur seuil :", round(meilleur_seuil, 2))
print("Meilleur F1 :", round(meilleur_f1, 4))

predictions_finales = (
    probabilites >= meilleur_seuil
).astype(int)

print(classification_report(y_validation, predictions_finales))

C:\Users\DELL\AppData\Local\Temp\ipykernel_2440\2279809485.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_categorielles = X.select_dtypes(include="object").columns.tolist()


0:	test: 0.6573219	best: 0.6573219 (0)	total: 26.7ms	remaining: 26.7s
100:	test: 0.7300678	best: 0.7333643 (67)	total: 3.9s	remaining: 34.7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7333642864
bestIteration = 67

Shrink model to first 68 iterations.
Meilleur seuil : 0.57
Meilleur F1 : 0.3214
              precision    recall  f1-score   support

           0       0.93      0.87      0.90       979
           1       0.26      0.41      0.32       110

    accuracy                           0.83      1089
   macro avg       0.60      0.64      0.61      1089
weighted avg       0.86      0.83      0.84      1089

